# CASR Community Detection ID2211 Project

Our project compares **Baseline Louvain** against the proposed **Consensus-Aware Selective Refinement (CASR)** algorithm.

CASR has two stages:

1. **Consensus-Based Stable Partition**  
   Run Louvain multiple times, build an edge-level consensus graph, then run weighted Louvain on the consensus graph.

2. **Instability-Aware Local Refinement**  
   Identify large, unstable communities and locally refine them using a higher resolution parameter.

We will be using the following two datasets:

- `email-Eu-core`
- `com-Amazon`

## 1. Imports

In [8]:
import time
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

## 2. Configuration and Parameters

Change the dataset settings here instead of using command-line arguments.

For `email-Eu-core`, use:

```python
DATASET = "email"
EDGE_FILE = "email-Eu-core.txt"
LABEL_FILE = "email-Eu-core-department-labels.txt"
COMMUNITY_FILE = None
```

For `com-Amazon`, use:

```python
DATASET = "amazon"
EDGE_FILE = "com-amazon.ungraph.txt"
LABEL_FILE = None
COMMUNITY_FILE = "com-amazon.top5000.cmty.txt"
```

In [22]:
# Choose either "email" or "amazon"
#DATASET = "email"
DATASET = "amazon"

#EDGE_FILE = "email-Eu-core.txt"
EDGE_FILE = "com-amazon.ungraph.txt"

#LABEL_FILE = "email-Eu-core-department-labels.txt"
LABEL_FILE = None

#COMMUNITY_FILE = None
COMMUNITY_FILE = "com-amazon.top5000.cmty.txt"

# CASR parameters
R = 20                    # number of Louvain runs for consensus
TAU = 0.3                 # consensus threshold
ALPHA = 3.0               # refinement strength
MIN_SIZE = 10             # minimum community size for refinement
STABILITY_THRESHOLD = 0.95
SEED = 0

# Stability evaluation
CASR_STABILITY_RUNS = 5   # use 3 for com-Amazon if runtime is high

## 3. Data Loading Functions

Both datasets are loaded as undirected graphs in order to keep the Louvain modularity comparison simple and consistent.

For `com-Amazon`, the ground-truth communities can overlap. Since NMI and ARI require one label per node, we assign each node to the first ground-truth community it appears in. This is a simplifying approximation for evaluation.

In [11]:
# Load the edge list as an undirected graph
def load_edges(path):
    G = nx.Graph()

    with open(path, "r") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue

            u, v = map(int, line.split()[:2])

            if u != v:
                G.add_edge(u, v)

    return G

# If email-Eu-core is chosen, load email-Eu-core department labels: node_id department_id
def load_email_labels(path):
    labels = {}

    with open(path, "r") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue

            node, label = map(int, line.split()[:2])
            labels[node] = label

    return labels


# Convert Amazon overlapping communities into one label per node. The first community containing a node is used as that node's label.  
def load_amazon_labels_from_communities(path):
    labels = {}

    with open(path, "r") as f:
        for cid, line in enumerate(f):
            if line.startswith("#") or not line.strip():
                continue

            nodes = list(map(int, line.split()))

            for node in nodes:
                if node not in labels:
                    labels[node] = cid

    return labels


# Use the above functions to load the graph and ground-truth labels depending on which dataset
def load_dataset(dataset, edge_file, label_file=None, community_file=None):
    G = load_edges(edge_file)
    labels = None

    if dataset == "email" and label_file is not None:
        labels = load_email_labels(label_file)
    elif dataset == "amazon" and community_file is not None:
        labels = load_amazon_labels_from_communities(community_file)

    return G, labels

## 4. Partition Helper Functions

These functions convert between these two common partition formats:

- `node -> community_id`
- `list of node sets`

In [12]:
# Convert list of node sets into node -> community_id
def communities_to_partition(communities):
    partition = {}

    for cid, community in enumerate(communities):
        for node in community:
            partition[node] = cid

    return partition


# Convert node -> community_id into list of node sets
def partition_to_communities(partition):
    groups = defaultdict(set)

    for node, cid in partition.items():
        groups[cid].add(node)

    return list(groups.values())


# Relabel community IDs to 0, 1, 2, ...
def relabel_partition(partition):
    mapping = {}
    new_partition = {}
    next_id = 0

    for node in sorted(partition):
        old = partition[node]

        if old not in mapping:
            mapping[old] = next_id
            next_id += 1

        new_partition[node] = mapping[old]

    return new_partition

## 5. Baseline Louvain

The baseline is standard Louvain modularity optimization.

In [13]:
# Run Louvain and return node -> community_id
def run_louvain(G, seed=0, resolution=1.0, weight="weight"):
    communities = nx.community.louvain_communities(
        G,
        seed=seed,
        resolution=resolution,
        weight=weight
    )

    return communities_to_partition(communities)


# Run Louvain multiple times using different random seeds (for CASR stage 1)
def run_louvain_many_times(G, R=10, seed=0):
    rng = random.Random(seed)
    partitions = []

    for _ in range(R):
        s = rng.randint(0, 10**9)
        partitions.append(run_louvain(G, seed=s))

    return partitions

## 6. CASR Stage 1: Consensus-Based Stable Partition

Stage 1 builds an edge-level consensus graph:


Only edges with consensus value at least `TAU` are retained. Their consensus values become edge weights.

In [14]:
# For each original graph edge (u, v), M_uv is the fraction of Louvain runs where u and v are assigned to the same community.
def build_edge_consensus(G, partitions):
    no_of_runs = len(partitions)
    consensus = {}

    for u, v in G.edges():
        count = 0

        for p in partitions:
            if p[u] == p[v]:
                count += 1

        consensus[(u, v)] = count / no_of_runs

    return consensus


# Build a weighted consensus graph after filtering away the weak consensus edges based on the tau threshold
def build_consensus_graph(G, consensus, tau=0.3):
    Gc = nx.Graph()
    Gc.add_nodes_from(G.nodes())

    for (u, v), mij in consensus.items():
        if mij >= tau:
            Gc.add_edge(u, v, weight=mij)

    return Gc


"""
    Stage 1:
        1. Run Louvain R times.
        2. Build edge-level consensus values.
        3. Build the consensus graph.
        4. Run weighted Louvain on the consensus graph.
"""
def stage1_consensus_partition(G, R=10, tau=0.3, seed=0):
    partitions = run_louvain_many_times(G, R=R, seed=seed)
    consensus = build_edge_consensus(G, partitions)
    Gc = build_consensus_graph(G, consensus, tau=tau)
    coarse_partition = run_louvain(Gc, seed=seed + 1, weight="weight")

    return coarse_partition, consensus, partitions, Gc

## 7. CASR Stage 2: Instability-Aware Local Refinement

Stage 2 computes the internal stability of each coarse community using the formula s(C) in our report


Large communities with low stability are refined locally using an adaptive resolution parameter found in our report


In [15]:
# Compute s(C): average consensus value over internal edges
def community_stability(G, nodes, consensus):
    sub_edges = list(G.subgraph(nodes).edges())

    if len(sub_edges) == 0:
        return 1.0

    values = []

    for u, v in sub_edges:
        if (u, v) in consensus:
            values.append(consensus[(u, v)])
        else:
            values.append(consensus.get((v, u), 0.0))

    return float(np.mean(values))


# Refine large communities whose internal stability is below the threshold.
def stage2_refinement(
    G,
    coarse_partition,
    consensus,
    alpha=2.0,
    min_size=10,
    stability_threshold=0.95,
    seed=0
):
    final_partition = dict(coarse_partition)
    communities = partition_to_communities(coarse_partition)

    next_cid = max(final_partition.values()) + 1
    num_candidates = 0
    num_refined = 0

    for idx, C in enumerate(communities):
        C = set(C)

        if len(C) < min_size:
            continue

        s_C = community_stability(G, C, consensus)

        if s_C >= stability_threshold:
            continue

        num_candidates += 1
        G_sub = G.subgraph(C).copy()

        if G_sub.number_of_edges() == 0:
            continue

        gamma_C = 1.0 + alpha * (1.0 - s_C)

        refined = run_louvain(
            G_sub,
            seed=seed + idx + 100,
            resolution=gamma_C,
            weight="weight"
        )

        refined_communities = partition_to_communities(refined)

        # Accept only if the local algorithm actually splits the community.
        if len(refined_communities) <= 1:
            continue

        num_refined += 1

        for sub_C in refined_communities:
            for node in sub_C:
                final_partition[node] = next_cid
            next_cid += 1

    return relabel_partition(final_partition), num_candidates, num_refined

## 8. Evaluation Functions

We evaluate using:

- **NMI** and **ARI** against ground truth labels
- **Modularity**
- **Number of detected communities**
- **Runtime**
- **Stability**, measured by average pairwise NMI across repeated runs

In [16]:
# Evaluate one partition using modularity, NMI, and ARI
def evaluate_partition(G, partition, labels=None):
    communities = partition_to_communities(partition)

    result = {
        "num_communities": len(communities),
        "modularity": nx.community.modularity(G, communities),
    }

    if labels is not None:
        common_nodes = sorted(set(partition) & set(labels))
        y_true = [labels[n] for n in common_nodes]
        y_pred = [partition[n] for n in common_nodes]

        result["NMI"] = normalized_mutual_info_score(y_true, y_pred)
        result["ARI"] = adjusted_rand_score(y_true, y_pred)
        result["labeled_nodes"] = len(common_nodes)
    else:
        result["NMI"] = np.nan
        result["ARI"] = np.nan
        result["labeled_nodes"] = 0

    return result


# Average pairwise NMI between repeated runs
def stability_score(partitions, nodes):
    scores = []

    for i in range(len(partitions)):
        for j in range(i + 1, len(partitions)):
            a = [partitions[i][n] for n in nodes]
            b = [partitions[j][n] for n in nodes]
            scores.append(normalized_mutual_info_score(a, b))

    return float(np.mean(scores)) if scores else np.nan

## 9. CASR Wrapper and CASR Stability

In order to measure CASR stability, we run the **entire CASR pipeline multiple times** and compute pairwise NMI between the final CASR outputs.

In [17]:
# Run full CASR once which returns the final partition
def run_casr_once(G, R=10, tau=0.3, alpha=1.0, min_size=10, stability_threshold=0.95, seed=0):
    
    coarse_partition, consensus, _, _ = stage1_consensus_partition(
        G,
        R=R,
        tau=tau,
        seed=seed
    )

    final_partition, _ , _ = stage2_refinement(
        G,
        coarse_partition,
        consensus,
        alpha=alpha,
        min_size=min_size,
        stability_threshold=stability_threshold,
        seed=seed
    )

    return final_partition


# Run CASR multiple times and compute average pairwise NMI to calculate CASR stability (Warning: Can take a long time !)
def casr_stability_score(
    G,
    num_runs=5,
    R=10,
    tau=0.5,
    alpha=1.0,
    min_size=30,
    stability_threshold=0.95,
    seed=0
):
    partitions = []

    for i in range(num_runs):
        p = run_casr_once(
            G,
            R=R,
            tau=tau,
            alpha=alpha,
            min_size=min_size,
            stability_threshold=stability_threshold,
            seed=seed + i * 100
        )
        partitions.append(p)

    return stability_score(partitions, sorted(G.nodes()))

## 10. Load Dataset

In [23]:
G, labels = load_dataset(
    DATASET,
    EDGE_FILE,
    label_file=LABEL_FILE,
    community_file=COMMUNITY_FILE
)

print(f"Dataset: {DATASET}")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Labeled nodes: {0 if labels is None else len(labels)}")

Dataset: amazon
Nodes: 334863
Edges: 925872
Labeled nodes: 16716


## 11. Run Baseline Louvain

In [24]:
print("Running baseline Louvain...")
start = time.time()
baseline_partition = run_louvain(G, seed=SEED)
baseline_time = time.time() - start

baseline_result = evaluate_partition(G, baseline_partition, labels)
baseline_result["method"] = "Baseline Louvain"
baseline_result["runtime_seconds"] = baseline_time

print("Measuring baseline Louvain stability...")
baseline_runs = run_louvain_many_times(G, R=R, seed=SEED)
baseline_result["stability_pairwise_NMI"] = stability_score(baseline_runs, sorted(G.nodes()))

baseline_result["refinement_candidates"] = np.nan
baseline_result["communities_refined"] = np.nan

pd.Series(baseline_result)

Running baseline Louvain...
Measuring baseline Louvain stability...


num_communities                        239
modularity                        0.926324
NMI                               0.843145
ARI                               0.330277
labeled_nodes                        16716
method                    Baseline Louvain
runtime_seconds                   32.82751
stability_pairwise_NMI            0.798946
refinement_candidates                  NaN
communities_refined                    NaN
dtype: object

## 12. Run CASR

In [25]:
print("Running CASR Stage 1 and Stage 2...")
start = time.time()

coarse_partition, consensus, _, consensus_graph = stage1_consensus_partition(
    G,
    R=R,
    tau=TAU,
    seed=SEED
)

final_partition, num_candidates, num_refined = stage2_refinement(
    G,
    coarse_partition,
    consensus,
    alpha=ALPHA,
    min_size=MIN_SIZE,
    stability_threshold=STABILITY_THRESHOLD,
    seed=SEED
)

casr_time = time.time() - start

casr_result = evaluate_partition(G, final_partition, labels)
casr_result["method"] = "CASR"
casr_result["runtime_seconds"] = casr_time
casr_result["refinement_candidates"] = num_candidates
casr_result["communities_refined"] = num_refined

print("Measuring CASR stability...")
casr_result["stability_pairwise_NMI"] = casr_stability_score(
    G,
    num_runs=CASR_STABILITY_RUNS,
    R=R,
    tau=TAU,
    alpha=ALPHA,
    min_size=MIN_SIZE,
    stability_threshold=STABILITY_THRESHOLD,
    seed=SEED
)

pd.Series(casr_result)

Running CASR Stage 1 and Stage 2...
Measuring CASR stability...


num_communities                  351
modularity                  0.927717
NMI                         0.869294
ARI                          0.40103
labeled_nodes                  16716
method                          CASR
runtime_seconds           637.745941
refinement_candidates              0
communities_refined                0
stability_pairwise_NMI      0.926766
dtype: object

## 13. Final Comparison Table

In [27]:
results = pd.DataFrame([baseline_result, casr_result])

columns = [
    "method",
    "NMI",
    "ARI",
    "modularity",
    "num_communities",
    "runtime_seconds",
    "stability_pairwise_NMI",
    "refinement_candidates",
    "communities_refined",
    "labeled_nodes",
]

results = results[columns]
results

,method,NMI,ARI,modularity,num_communities,runtime_seconds,stability_pairwise_NMI,refinement_candidates,communities_refined,labeled_nodes
0,Baseline Louvain,0.843145,0.330277,0.926324,239,32.827510,0.798946,NaN,NaN,16716
1,CASR,0.869294,0.401030,0.927717,351,637.745941,0.926766,0.0,0.0,16716


## 14. Save Results

In [ ]:
output_file = f"results_{DATASET}.csv"
results.to_csv(output_file, index=False)
print(f"Saved results to: {output_file}")

## 15. Suggested Interpretation

Use the following ideas when discussing results:

- If **NMI/ARI improve but modularity decreases**, this can still support the project motivation. Louvain directly optimizes modularity, but modularity can merge smaller meaningful communities due to the resolution limit.
- If **CASR stability is higher**, then the consensus step is successfully reducing sensitivity to random seeds.
- If `refinement_candidates` and `communities_refined` are both zero, then Stage 2 did not activate. Increase `STABILITY_THRESHOLD` or lower `MIN_SIZE`.
- If runtime is too high on `com-Amazon`, reduce `R` or `CASR_STABILITY_RUNS`.